<a href="https://colab.research.google.com/github/jsk900210-oss/AIFFEL_Quest_ENG/blob/master/%EC%B1%97%EB%B4%87_%EA%B0%9C%EC%84%A0_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 멋진 챗봇 만들기 — 개선 버전 (Pre-LN final_norm 수정)

**개선점**
1. 중복 제거 완화 → 데이터 약 3.3만 (루브릭 '3만 가량' 충족)
2. 패딩 마스크 수정(`supports_masking=True`) → 어텐션이 `<pad>` 무시
3. **[신규] Pre-LN final_norm 추가** → Encoder/Decoder 스택 마지막 출력에 LayerNorm 추가
   - 문제: Pre-LN 구조에서 residual connection이 누적되면서 출력 norm 폭주
   - 증상: cross-attention의 Q·K^T 값이 극단적으로 커져 softmax가 한 위치에 100% 집중 → 모드 붕괴
   - 해결: Encoder·Decoder 스택 끝에 `LayerNormalization` 한 줄 추가 (Pre-LN 표준 구조)

※ 상단 메뉴 **런타임 → 모두 실행(Run all)** 으로 위→아래 순서대로 실행하세요.

In [1]:
!pip install -q kiwipiepy gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 82.9 MB/s eta 0:00:00


In [2]:
# Step 1. 데이터 로드 (jsDelivr 미러 → 429/빈파일 회피)
import pandas as pd
url = 'https://cdn.jsdelivr.net/gh/songys/Chatbot_data@master/ChatbotData.csv'
data = pd.read_csv(url)
questions = list(data['Q'])
answers   = list(data['A'])
print("원본:", len(questions))

원본: 11823


In [3]:
# Step 2. Kiwi 토크나이저 (MeCab 대체, 설치 간단)
from kiwipiepy import Kiwi
kiwi = Kiwi()
def kiwi_morphs(sentence):
    return [t.form for t in kiwi.tokenize(sentence)]
print(kiwi_morphs("안녕하세요 반갑습니다"))

['안녕', '하', '세요', '반갑', '습니다']


In [4]:
# Step 3. 정제 + 토큰화  (중복 제거 완화: 완전 동일한 (질문,답변) 쌍만 제거)
import re
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"[^가-힣a-z0-9?.!,]+", " ", sentence)
    return sentence.strip()

def build_corpus(src_data, tgt_data, tokenizer, max_len=40):
    src_corpus, tgt_corpus = [], []
    seen = set()
    for src, tgt in zip(src_data, tgt_data):
        src_p, tgt_p = preprocess_sentence(src), preprocess_sentence(tgt)
        key = (src_p, tgt_p)
        if key in seen:            # 완전 동일한 쌍만 제거 (데이터 더 확보)
            continue
        src_tok, tgt_tok = tokenizer(src_p), tokenizer(tgt_p)
        if len(src_tok) > max_len or len(tgt_tok) > max_len:
            continue
        seen.add(key)
        src_corpus.append(src_tok); tgt_corpus.append(tgt_tok)
    return src_corpus, tgt_corpus

que_corpus, ans_corpus = build_corpus(questions, answers, kiwi_morphs)
print("중복 제거 후:", len(que_corpus), '|', que_corpus[0], '→', ans_corpus[0])

중복 제거 후: 11747 | ['12', '시', '땡', '!'] → ['하루', '가', '또', '가', '네요', '.']


In [5]:
# Step 4. 데이터 증강 (Lexical Substitution) → 3배 (약 3.3만)
from gensim.models import Word2Vec
import random
w2v = Word2Vec(sentences=que_corpus + ans_corpus, vector_size=100, window=5, min_count=1, sg=1, workers=4)

def lexical_sub(tokens, model, threshold=0.7):
    aug = tokens.copy()
    cand = [i for i, t in enumerate(aug) if t in model.wv]
    if not cand: return aug
    idx = random.choice(cand)
    sims = model.wv.most_similar(aug[idx], topn=1)
    if sims and sims[0][1] >= threshold: aug[idx] = sims[0][0]
    return aug

que_aug = [lexical_sub(q, w2v) for q in que_corpus]
ans_aug = [lexical_sub(a, w2v) for a in ans_corpus]
src_corpus = que_corpus + que_aug   + que_corpus
tgt_corpus = ans_corpus + ans_corpus + ans_aug
print("증강 후:", len(src_corpus), len(tgt_corpus))

증강 후: 35241 35241


In [6]:
# Step 5. 벡터화 (<start>/<end> + 공유 단어사전)
import tensorflow as tf
tgt_corpus = [a if a and a[0]=='<start>' else ['<start>']+a+['<end>'] for a in tgt_corpus]
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='', oov_token='<unk>')
tokenizer.fit_on_texts(src_corpus + tgt_corpus)
enc_train = tokenizer.texts_to_sequences(src_corpus)
dec_train = tokenizer.texts_to_sequences(tgt_corpus)
enc_train = tf.keras.preprocessing.sequence.pad_sequences(enc_train, padding='post')
dec_train = tf.keras.preprocessing.sequence.pad_sequences(dec_train, padding='post')
VOCAB_SIZE = len(tokenizer.index_word) + 1
print("어휘:", VOCAB_SIZE, "| enc:", enc_train.shape, "| dec:", dec_train.shape)

어휘: 4992 | enc: (35241, 31) | dec: (35241, 42)


In [7]:
# Step 6. Transformer 정의
# [수정] Pre-LN final_norm 추가
# - 문제: Encoder/Decoder 스택 마지막 출력에 LayerNorm 누락
# - 원인: residual connection 누적 → 출력 norm 폭주 → cross-attention Q·K^T 극단적 증가
#         → softmax 모드 붕괴 (한 위치에 100% attention 집중, 답변 단조로워짐)
# - 해결: Encoder / Decoder 클래스 끝에 self.final_norm = LayerNormalization() 추가
#         forward 마지막에 out = self.final_norm(out) 통과 → Pre-LN 표준 구조 완성
import tensorflow as tf, numpy as np

def positional_encoding(length, depth):
    d=depth/2; pos=np.arange(length)[:,None]; dep=np.arange(d)[None,:]/d
    ang=pos*(1/(10000**dep))
    return tf.cast(np.concatenate([np.sin(ang),np.cos(ang)],-1),tf.float32)

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(s,vocab,d):
        super().__init__(); s.d=d
        s.emb=tf.keras.layers.Embedding(vocab,d,mask_zero=True)
        s.pe=positional_encoding(2048,d)
    def compute_mask(s,*a,**k): return s.emb.compute_mask(*a,**k)
    def call(s,x):
        L=tf.shape(x)[1]; x=s.emb(x)*tf.math.sqrt(tf.cast(s.d,tf.float32))
        return x+s.pe[None,:L,:]

class FF(tf.keras.layers.Layer):
    def __init__(s,d,dff,dr):
        super().__init__(); s.supports_masking=True
        s.seq=tf.keras.Sequential([tf.keras.layers.Dense(dff,activation='relu'),tf.keras.layers.Dense(d),tf.keras.layers.Dropout(dr)])
        s.add=tf.keras.layers.Add(); s.ln=tf.keras.layers.LayerNormalization()
    def call(s,x): return s.ln(s.add([x,s.seq(x)]))

def mha(nh,d,dr): return tf.keras.layers.MultiHeadAttention(num_heads=nh,key_dim=d//nh,dropout=dr)

class EncLayer(tf.keras.layers.Layer):
    def __init__(s,d,nh,dff,dr):
        super().__init__(); s.supports_masking=True
        s.a=mha(nh,d,dr); s.add=tf.keras.layers.Add(); s.ln=tf.keras.layers.LayerNormalization(); s.ff=FF(d,dff,dr)
    def call(s,x): x=s.ln(s.add([x,s.a(x,x,x)])); return s.ff(x)

class DecLayer(tf.keras.layers.Layer):
    def __init__(s,d,nh,dff,dr):
        super().__init__(); s.supports_masking=True
        s.sa=mha(nh,d,dr); s.ca=mha(nh,d,dr)
        s.a1=tf.keras.layers.Add(); s.a2=tf.keras.layers.Add()
        s.l1=tf.keras.layers.LayerNormalization(); s.l2=tf.keras.layers.LayerNormalization(); s.ff=FF(d,dff,dr)
    def call(s,x,ctx):
        x=s.l1(s.a1([x,s.sa(x,x,x,use_causal_mask=True)]))
        x=s.l2(s.a2([x,s.ca(x,ctx,ctx)])); return s.ff(x)

class Transformer(tf.keras.Model):
    def __init__(s,nl,d,nh,dff,vocab,dr):
        super().__init__()
        s.ep=PositionalEmbedding(vocab,d); s.dp=PositionalEmbedding(vocab,d)
        s.encs=[EncLayer(d,nh,dff,dr) for _ in range(nl)]
        s.decs=[DecLayer(d,nh,dff,dr) for _ in range(nl)]
        s.drop=tf.keras.layers.Dropout(dr)
        # ✅ [수정] Pre-LN 표준 구조: Encoder/Decoder 스택 끝에 final_norm 추가
        # 이미지 코드 참조: self.final_norm = nn.LayerNorm(d_model, eps=1e-6)
        # TF 버전: LayerNormalization(epsilon=1e-6)
        s.enc_final_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)  # ← Encoder stack 끝 정규화
        s.dec_final_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)  # ← Decoder stack 끝 정규화
        s.final=tf.keras.layers.Dense(vocab)

    def call(s,inp):
        ctx,x=inp
        ctx=s.drop(s.ep(ctx))
        for l in s.encs: ctx=l(ctx)
        # ✅ [수정] Encoder 스택 마지막 출력 정규화
        # 이미지 코드: out = self.final_norm(out)  # ← stack 끝나고 정규화
        ctx = s.enc_final_norm(ctx)  # ← residual 누적으로 인한 norm 폭주 방지

        x=s.drop(s.dp(x))
        for l in s.decs: x=l(x,ctx)
        # ✅ [수정] Decoder 스택 마지막 출력 정규화
        # 이미지 코드: out = self.final_norm(out)  # ← stack 끝나고 정규화
        x = s.dec_final_norm(x)     # ← softmax 모드 붕괴 방지 핵심

        y=s.final(x)
        try: del y._keras_mask
        except: pass
        return y

print("Transformer 정의 완료 (Pre-LN final_norm 수정 적용)")

Transformer 정의 완료 (Pre-LN final_norm 수정 적용)


In [8]:
# Step 7. 훈련 (과적합 방지: dropout 0.2 / 경량 모델 / warmup)
n_layers, d_model, n_heads, d_ff, dropout = 1, 368, 8, 1024, 0.2
transformer = Transformer(n_layers, d_model, n_heads, d_ff, VOCAB_SIZE, dropout)

class Sched(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(s,d,w=1000): super().__init__(); s.d=tf.cast(d,tf.float32); s.w=w
    def __call__(s,step):
        step=tf.cast(step,tf.float32)
        return tf.math.rsqrt(s.d)*tf.math.minimum(tf.math.rsqrt(step), step*(s.w**-1.5))

opt=tf.keras.optimizers.Adam(Sched(d_model,1000),beta_1=0.9,beta_2=0.98,epsilon=1e-9)
def masked_loss(y,p):
    m=tf.cast(y!=0,tf.float32)
    l=tf.keras.losses.sparse_categorical_crossentropy(y,p,from_logits=True)
    return tf.reduce_sum(l*m)/tf.reduce_sum(m)
def masked_acc(y,p):
    p=tf.argmax(p,-1); y=tf.cast(y,p.dtype); m=tf.cast(y!=0,tf.float32)
    return tf.reduce_sum(tf.cast(y==p,tf.float32)*m)/tf.reduce_sum(m)
transformer.compile(loss=masked_loss, optimizer=opt, metrics=[masked_acc])

dec_input, dec_target = dec_train[:, :-1], dec_train[:, 1:]
transformer.fit((enc_train, dec_input), dec_target, batch_size=64, epochs=10)

Epoch 1/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - loss: 3.7773 - masked_acc: 0.3856
Epoch 2/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - loss: 2.0449 - masked_acc: 0.5727
Epoch 3/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - loss: 1.2535 - masked_acc: 0.7150
Epoch 4/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - loss: 0.7442 - masked_acc: 0.8211
Epoch 5/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 41ms/step - loss: 0.5083 - masked_acc: 0.8743
Epoch 6/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - loss: 0.3912 - masked_acc: 0.9026
Epoch 7/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - loss: 0.3241 - masked_acc: 0.9190
Epoch 8/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - loss: 0.2838 - masked_acc: 0.9279
Epoch 9/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - loss: 0.2547 - masked_acc: 0.9350
Epoch 10/10
551/551 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - loss: 0.2338 - masked_acc: 0.9392


In [9]:
# Step 8. 예문 답변 생성 (제출용 사례)
def translate(sentence, max_len=40):
    tokens = kiwi_morphs(preprocess_sentence(sentence))
    enc_in = tokenizer.texts_to_sequences([tokens])
    enc_in = tf.keras.preprocessing.sequence.pad_sequences(enc_in, maxlen=enc_train.shape[1], padding='post')
    enc_in = tf.constant(enc_in)
    start, end = tokenizer.word_index['<start>'], tokenizer.word_index['<end>']
    output = [start]
    for _ in range(max_len):
        preds = transformer((enc_in, tf.constant([output])), training=False)
        nid = int(tf.argmax(preds[0, -1]))
        if nid == end: break
        output.append(nid)
    return ' '.join(tokenizer.index_word.get(i,'') for i in output[1:])

examples = ["지루하다, 놀러가고 싶어.","오늘 일찍 일어났더니 피곤하다.",
            "간만에 여자친구랑 데이트 하기로 했어.","집에 있는다는 소리야."]
for q in examples:
    print(q, "→", translate(q))

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (1, 8, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


지루하다, 놀러가고 싶어. → 같이 가 ᆯ 수 있 을 거 이 예요 .
오늘 일찍 일어났더니 피곤하다. → 정신 차리 세요 .
간만에 여자친구랑 데이트 하기로 했어. → 데이트 로 하 기 좋 죠 .
집에 있는다는 소리야. → 세상 에 별별 사람 들 이 다 있 는 거 이 예요 .


In [10]:
# Step 9. BLEU (참고 — 챗봇은 정답이 다양해 낮게 나오는 게 정상)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smooth = SmoothingFunction().method1
def calculate_bleu(question, reference_answer):
    pred = translate(question).split()
    ref  = kiwi_morphs(preprocess_sentence(reference_answer))
    return sentence_bleu([ref], pred, smoothing_function=smooth)
for i in range(3):
    print(questions[i], "→", translate(questions[i]),
          "| BLEU:", round(calculate_bleu(questions[i], answers[i]),3))

12시 땡! → 하루 가 또 가 죠 . | BLEU: 0.537
1지망 학교 떨어졌어 → 위로 하 어 드리 ᆸ니다 . | BLEU: 1.0
3박4일 놀러가고 싶다 → 여행 은 언제나 좋 죠 . | BLEU: 1.0
